# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/refactor/dynamics-lam-overhaul/notebooks/colab_training_and_demo.ipynb)

This notebook executes the end-to-end pipeline of **`latent-play`** on Google Colab Pro (**A100 GPU + High-RAM**) with **automatic checkpoint persistence to Google Drive every 5 minutes**:
- If Colab disconnects at any point, clicking **Run all** will automatically resume training from the latest checkpoint saved in your Google Drive (`MyDrive/latent_play_overhaul/`).
- Datasets (`data/`) are stored on Colab's local NVMe disk for maximum read throughput (`np.memmap`), while `checkpoints/` and `outputs/` are symlinked directly to your Google Drive.

## 1. Setup, Google Drive Mounting, and Repository Cloning

In [1]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

CUDA Available: True
Device: NVIDIA A100-SXM4-40GB
VRAM: 42.41 GB
BF16 Supported: True


In [2]:
# 1. Cleanly clone repository strictly on refactor/dynamics-lam-overhaul
import os
BRANCH = "refactor/dynamics-lam-overhaul"
!rm -rf /content/latent-play
!git clone -b {BRANCH} https://github.com/rivas-vega/latent-play.git /content/latent-play
%cd /content/latent-play
!git branch -v
os.environ["PYTHONPATH"] = "/content/latent-play"

# 2. Mount Google Drive and symlink to a clean isolated directory
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/latent_play_overhaul"
    os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    for folder in ("checkpoints", "outputs"):
        if os.path.islink(folder):
            os.unlink(folder)
        elif os.path.isdir(folder):
            !rm -rf {folder}
        os.symlink(f"{DRIVE_ROOT}/{folder}", folder)
    print(f"Checkpoints and outputs symlinked to Google Drive: {DRIVE_ROOT}")
except Exception as e:
    print(f"Running without Google Drive ({e}): saving to local disk.")

# 3. Install dependencies and run the 47 unit tests
!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q


Cloning into '/content/latent-play'...
remote: Enumerating objects: 570, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 570 (delta 52), reused 59 (delta 31), pack-reused 464 (from 1)
Receiving objects: 100% (570/570), 27.49 MiB | 19.67 MiB/s, done.
Resolving deltas: 100% (294/294), done.
/content/latent-play
* refactor/dynamics-lam-overhaul 0a66010 feat(planning): support cost_metric options, horizon ablation, and Bayes-ceiling relative probe metrics
Running without Google Drive (mount failed): saving to local disk.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 12.6 MB/s eta 0:00

## 2. Crafter Experiment: Quick Validation vs. Full Benchmark
Crafter is the primary open-world survival benchmark for `latent-play`.
Select `fast_validation` for a ~12 min end-to-end smoke test, or `full_benchmark` for the final 30K-step paper run.

In [3]:
# @title Select Crafter Experiment Scale
# @markdown Choose between a fast 12-minute smoke test or the full 30k-step benchmark.
CRAFTER_MODE = "fast_validation"  # @param ["fast_validation", "full_benchmark"]

if CRAFTER_MODE == "fast_validation":
    CRAFTER_STEPS = 50000
    TRAIN_STEPS = 2500
    WARMUP_STEPS = 250
    EVAL_EVERY = 500
    SWEEP_EPISODES = 10
    SWEEP_SEEDS = "0"
    SWEEP_FRACTIONS = "0.01 0.05 0.1 1.0"
else:
    CRAFTER_STEPS = 300000
    TRAIN_STEPS = 30000
    WARMUP_STEPS = 1000
    EVAL_EVERY = 2000
    SWEEP_EPISODES = 30
    SWEEP_SEEDS = "0 1 2"
    SWEEP_FRACTIONS = "0.001 0.01 0.05 0.1 1.0"

print(f"Active Crafter Mode: {CRAFTER_MODE}")
print(f"Data steps: {CRAFTER_STEPS} | Model train steps: {TRAIN_STEPS} | Eval episodes: {SWEEP_EPISODES}")


Active Crafter Mode: fast_validation
Data steps: 50000 | Model train steps: 2500 | Eval episodes: 10


In [4]:
!if [ ! -f data/crafter/meta.json ]; then python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps {CRAFTER_STEPS} --workers 4; fi
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide a

## 3. Train Latent Action Model (LAM) on `Crafter` (`label_fraction=0.0`)
Automatically saves checkpoints every 5 minutes to your Google Drive (`MyDrive/latent_play_overhaul/checkpoints/lam_crafter`).

In [5]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml --force train.compile=true train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


device=cuda  train windows=46939  val windows=2508
LAM params: 31.39M
--force specified: discarding checkpoints in checkpoints/lam_crafter and training from step 0.
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0] Graph break from `Tensor.item()`, consider setting:
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0]     torch._dynamo.config.capture_scalar_outputs = True
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0] or:
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0] to include these operations in the captured graph.
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0] 
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0] Graph break: from user code at:
W1019 20:02:10.673000 3860 torch/_dynamo/variables/tensor.py:1379] [3/0]   File "/co

## 4. Train JEPA Dynamics + SIGReg on `Crafter` (Latent Actions vs. Oracle)
Automatically saves checkpoints every 5 minutes to your Google Drive (`dyn_crafter_lam` and `dyn_crafter_oracle`).

In [6]:
# 5a. Our model (latent-play): conditioned purely on discovered LAM latent codes
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}

# 5b. Action Oracle (Upper Bound): conditioned on 100% ground-truth Crafter actions
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


device=cuda  train windows=43546  val windows=2313
loading pretrained LAM from checkpoints/lam_crafter/lam_final.pt...
Dynamics conditioned on 8 latent actions!
--force specified: discarding checkpoints in checkpoints/dyn_crafter_lam and training from step 0.
time=37.4  step=100  loss=0.1563  pred_loss=0.0901  var_loss=0  cov_loss=0  sigreg_loss=0.2637  eff_rank=85  tau=0.996  gnorm=7.996  lr=0.00012  it_s=2.68  roll_loss=0.106  dec_loss=0.03889
time=73.4  step=200  loss=0.1528  pred_loss=0.09062  var_loss=0  cov_loss=0  sigreg_loss=0.2192  eff_rank=80.17  tau=0.9961  gnorm=4.562  lr=0.00024  it_s=2.787  roll_loss=0.1025  dec_loss=0.0259
time=109.4  step=300  loss=0.131  pred_loss=0.07958  var_loss=0  cov_loss=0  sigreg_loss=0.2011  eff_rank=76.68  tau=0.9961  gnorm=3.568  lr=0.0002997  it_s=2.785  roll_loss=0.08268  dec_loss=0.01943
time=145.4  step=400  loss=0.1124  pred_loss=0.06686  var_loss=0  cov_loss=0  sigreg_loss=0.2355  eff_rank=77.71  tau=0.9962  gnorm=2.638  lr=0.0002969  i

## 5. Label Fraction Sweep (0.1% to 100%), CEM Planning, and VoE Physical Scorer


In [7]:
!python scripts/sweep_labels.py \
    --use-state-context \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions {SWEEP_FRACTIONS} \
    --num-episodes {SWEEP_EPISODES} \
    --seeds {SWEEP_SEEDS} \
    --cem-horizon 4 \n    --cem-cost-metric l1 \n    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures


usage: sweep_labels.py [-h] [--data-root DATA_ROOT] [--lam-ckpt LAM_CKPT]
                       [--dyn-ckpt DYN_CKPT] [--oracle-ckpt ORACLE_CKPT]
                       [--fractions FRACTIONS [FRACTIONS ...]]
                       [--num-episodes NUM_EPISODES]
                       [--target-dist TARGET_DIST] [--max-steps MAX_STEPS]
                       [--out-dir OUT_DIR] [--seeds SEEDS [SEEDS ...]]
                       [--use-state-context] [--cem-samples CEM_SAMPLES]
                       [--cem-elites CEM_ELITES] [--cem-iters CEM_ITERS]
                       [--cem-horizon CEM_HORIZON]
                       [--cem-cost-metric {auto,spatial_conv,l1,cosine,pooled}]
                       [--collapse-blocked] [--distractor]
sweep_labels.py: error: unrecognized arguments: n n
VoE [teleport     ] | Plausible: 0.0301 | Violated: 0.2812 | Ratio: 10.13x
VoE [disappearance] | Plausible: 0.0301 | Violated: 0.2879 | Ratio: 10.39x
VoE [contradiction] | Plausible: 0.0301 | Violated: 0

### 5b. Systematic Ablation Study: Cost Metric and Horizon
Compare CEM planning performance across cost metrics (spatial_conv, l1, cosine, pooled) and horizons (H in {2, 3, 4, 6}).

In [8]:
# Run systematic planning ablations on the Oracle dynamics model
!python scripts/eval_ablations.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --num-episodes 10 \
    --out-dir outputs/ablations_oracle

# Run systematic planning ablations on our Latent-Play dynamics model
!python scripts/eval_ablations.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --alignment-checkpoint outputs/sweep_results/alignment_f0.01_s0.pt \
    --num-episodes 10 \
    --out-dir outputs/ablations_lam


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
ABLATION 1: CEM COST METRIC (fixing Horizon H=4)
Cost Metric: spatial_conv    | Success:  10.0% | Avg Steps: 11.00
Cost Metric: l1              | Success:  20.0% | Avg Steps: 8.50
Cost Metric: cosine          | Success:  10.0% | Avg Steps: 3.00
Cost Metric: pooled          | Success:  10.0% | Avg Steps: 7.00

ABLATION 2: PLANNING HORIZON H (using best cost metric: l1)
Horizon H: 2                | Success:  20.0% | Avg Steps: 5.50
Horizon H: 3                | Success:  10.0% | Avg Steps: 6.00
Horizon H: 4                | Success:  10.0% | Avg Steps: 6.00
Horizon H: 6                | Success:   0.0% | Avg Steps: nan

Saved abla

In [9]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --align-ckpt outputs/sweep_results/alignment_f0.01_s0.pt \
    --target-dist 4 \
    --num-episodes {SWEEP_EPISODES}

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")


loaded dynamics from checkpoints/dyn_crafter_lam/dyn_final.pt (num_actions=8)
Traceback (most recent call last):
  File "/content/latent-play/scripts/eval_planning.py", line 220, in <module>
    main()
    ~~~~^^
  File "/content/latent-play/scripts/eval_planning.py", line 156, in main
    align_ckpt = load_checkpoint(args.alignment_checkpoint, device)
  File "/content/latent-play/latent_play/utils/ckpt.py", line 45, in load_checkpoint
    return torch.load(path, map_location=map_location, weights_only=False)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __

FileNotFoundError: No such file or directory: 'outputs/eval_planning/planned_trajectory.gif'

FileNotFoundError: No such file or directory: 'outputs/eval_planning/planned_trajectory.gif'

<IPython.core.display.Image object>

## 6. Package Results and Logs for Download


In [10]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")

  adding: outputs/ (stored 0%)
  adding: outputs/.gitkeep (stored 0%)
  adding: outputs/figures/ (stored 0%)
  adding: outputs/figures/fig1_label_efficiency.pdf (deflated 31%)
  adding: outputs/figures/fig3_codebook_purity.pdf (deflated 32%)
  adding: outputs/figures/fig1_label_efficiency.png (deflated 9%)
  adding: outputs/figures/fig2_anticollapse_rollouts.png (deflated 9%)
  adding: outputs/figures/fig3_codebook_purity.png (deflated 11%)
  adding: outputs/figures/fig2_anticollapse_rollouts.pdf (deflated 31%)
  adding: outputs/ablations_oracle/ (stored 0%)
  adding: outputs/ablations_oracle/ablation_results.json (deflated 79%)
  adding: outputs/crafter_samples.gif (deflated 0%)
  adding: outputs/voe_metrics.json (deflated 58%)
  adding: checkpoints/dyn_crafter_lam/log.jsonl (deflated 68%)
  adding: checkpoints/dyn_crafter_oracle/log.jsonl (deflated 68%)
  adding: checkpoints/lam_crafter/log.jsonl (deflated 64%)
  adding: checkpoints/lam_crafter/confusion.json (deflated 54%)
  adding:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>